## Plotting inference results

In [ ]:
from typing import List, Dict, Any, Tuple, Optional
import os

import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
from netin.models import PATCHModel, CompoundLFM

from patch.constants import PATH_INFERENCE, MAP_LFM_SHORT, SIZE_FIG, MAP_MODEL_COLOR, MAP_LFM_SHORT, APS, DBLP, PATH_PLOTS, MAP_DATA_COLOR, MAP_STAT_LABEL, N_SAMPLES

In [ ]:
# Set figure size
plt.rcParams["figure.figsize"] = SIZE_FIG
# Remove legend border
plt.rcParams["legend.frameon"] = False
# Remove top and right axis
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
# Set font size
plt.rcParams["font.size"] = 10

In [ ]:
PATH_PREFIX = "7r-smc_"
LFM_COMBS = [
    (CompoundLFM.HOMOPHILY.value, CompoundLFM.UNIFORM.value),
    (CompoundLFM.HOMOPHILY.value, CompoundLFM.HOMOPHILY.value),
    (CompoundLFM.PAH.value, CompoundLFM.UNIFORM.value),
    (CompoundLFM.PAH.value, CompoundLFM.PAH.value),
]
DECADE = 1980
L_DECADES_APS = [1970, 1980, 1990, 2000, 2010]
L_DECADES_DBLP = [1970, 1980, 1990, 2000]
N_BINS = 15
DURATION = 10

In [ ]:
def create_folder_name(source: str, lfm_global: str, lfm_tc: str, decade: int = DECADE, prefix: str = PATH_PREFIX) -> str:
    return os.path.join(
        "..",
        PATH_INFERENCE,
        source if source == DBLP else "",
        f"{prefix}lfm-g-{lfm_global}_lfm-t-{lfm_tc}_d-{decade}/")

def create_posterior_file_name(folder_name: str) -> str:
    return os.path.join(folder_name, "posteriors.npz")

In [ ]:
folder_plots = os.path.join("../", PATH_PLOTS, PATH_PREFIX, "empirical")
if not os.path.exists(folder_plots):
    os.makedirs(folder_plots)
print(f"Saving plots to {folder_plots}")

In [ ]:
np_file = np.load(
    create_posterior_file_name(
        create_folder_name(
            source=APS,
            lfm_global=CompoundLFM.HOMOPHILY.value,
            lfm_tc=CompoundLFM.HOMOPHILY.value,
            decade=DECADE)))
print(np_file.files)

In [ ]:
np_file['f_m']

## Evolution of f_m

In [ ]:
def plot_fm(
    tpl_fig_ax: Optional[Tuple[plt.Figure, plt.Axes]] = None,
    is_subplot: bool = True)\
        -> Tuple[plt.Figure, plt.Axes]:
    fig, ax = tpl_fig_ax if tpl_fig_ax is not None else plt.subplots()
    l_fm_aps, l_fm_dblp = [], []
    for source, l_decades, l_fm in zip([APS, DBLP], [L_DECADES_APS, L_DECADES_DBLP], [l_fm_aps, l_fm_dblp]):
        for decade in l_decades:
            np_file = np.load(
                create_posterior_file_name(
                    create_folder_name(
                        source=source,
                        lfm_global=CompoundLFM.HOMOPHILY.value,
                        lfm_tc=CompoundLFM.HOMOPHILY.value,
                        decade=decade)))
            l_fm.append(np_file['f_m'])

    ax.plot(
        L_DECADES_APS,
        l_fm_aps,
        marker="o",
        color=MAP_DATA_COLOR[APS](.85),
        label=str.upper(APS))
    ax.plot(
        L_DECADES_DBLP,
        l_fm_dblp,
        marker="o",
        color=MAP_DATA_COLOR[DBLP](.85),
        label=str.upper(DBLP))

    # Format y-tick labels to percentage
    ax.set_yticks(
        [.05, .1],
        labels=["{:.0%}".format(x) for x in [.05, .1]])

    ax.set_xlabel(
        "Decade",
        # Bring closer to axis
        labelpad=-.5
    )
    ax.set_ylabel(
        "$f_w$",
        # Bring closer to axis
        labelpad=-5)
    if not is_subplot:
        _ = ax.legend()
    return fig, ax

In [ ]:
_ = plot_fm(is_subplot=False)

In [ ]:
# Iterate over decades from 1970 to 2000
results_aps = {combo: [] for combo in LFM_COMBS}
results_dblp = {combo: [] for combo in LFM_COMBS}
for source, l_decades, results in zip(
        [APS, DBLP],
        [L_DECADES_APS, L_DECADES_DBLP],
        [results_aps, results_dblp]):
    for decade in l_decades:
        for lfm_global, lfm_tc in LFM_COMBS:
            file_name = create_posterior_file_name(
                create_folder_name(
                    source=source,
                    lfm_global=lfm_global,
                    lfm_tc=lfm_tc,
                    decade=decade))
            with np.load(file_name) as data:
                results[
                    (lfm_global, lfm_tc)].append(
                        np.quantile(data['discrepancies'], q=(0.05, 0.5, 0.95)))

In [ ]:
def plot_model_selection(
    tpl_fig_ax: Optional[Tuple[plt.Figure, plt.Axes]] = None,
    is_subplot: bool = True)\
        -> Tuple[plt.Figure, plt.Axes]:
    # Plotting
    l_artists = []
    fig, a_ax = tpl_fig_ax\
        if tpl_fig_ax is not None else\
            plt.subplots(ncols=2, sharey=True, sharex=True)
    for i, (results, l_decades) in enumerate(zip([results_dblp, results_aps], [L_DECADES_DBLP, L_DECADES_APS])):
        for (lfm_global, lfm_tc), values in results.items():
            values = np.array(values)
            artist = a_ax[i].errorbar(
                l_decades,
                values[:, 1],
                yerr=[values[:, 1] - values[:, 0], values[:, 2] - values[:, 1]],
                fmt='o-',
                color=MAP_MODEL_COLOR[(lfm_global, lfm_tc)],
                label=f"${MAP_LFM_SHORT[lfm_global]},{MAP_LFM_SHORT[lfm_tc]}$" if i == 0 else None)
            l_artists.append(artist)

            if i == 0:
                # Plot model labels to the right of last markers
                y_off = 1.25\
                    if (lfm_global, lfm_tc) == (CompoundLFM.PAH.value, CompoundLFM.PAH.value)\
                        else -1.25\
                            if (lfm_global, lfm_tc) == (CompoundLFM.HOMOPHILY.value, CompoundLFM.HOMOPHILY.value)\
                                else 0
                a_ax[i].text(
                    L_DECADES_DBLP[-1] + 1.5,
                    values[-1, 1] + y_off,
                    f"{MAP_LFM_SHORT[lfm_global]},{MAP_LFM_SHORT[lfm_tc]}",
                    color=MAP_MODEL_COLOR[(lfm_global, lfm_tc)],
                    verticalalignment='center',
                    transform=a_ax[i].transData,
                    # Bold if PAH, U
                    fontweight='bold'\
                        if ((lfm_global == CompoundLFM.PAH.value) and (lfm_tc == CompoundLFM.UNIFORM.value))\
                        else 'normal',
                    # Italic
                    fontstyle='italic')


    for source, ax in zip((DBLP, APS), a_ax):
        ax.set_xlabel("Decade",
                      labelpad=-.5)
        # Set source at top left
        ax.text(0.1, .95, str.upper(source), transform=ax.transAxes)

    # Show legend right to plot
    if not is_subplot:
        a_ax[0].set_ylabel('Distance')
        _ = fig.legend(
            loc='center left',
            bbox_to_anchor=(.975, 0.5),
            # Reduce line length
            # Reduce line width
            handlelength=1.25,
            labelspacing=0.25)
    return fig, a_ax, l_artists

In [ ]:
def plot_model_selection_bar(
    horizontal: bool = True,
    tpl_fig_ax: Optional[Tuple[plt.Figure, plt.Axes]] = None,
    is_subplot: bool = True)\
        -> Tuple[plt.Figure, plt.Axes]:
    # Plotting
    l_artists = []
    fig, a_ax = tpl_fig_ax\
        if tpl_fig_ax is not None else\
            plt.subplots(ncols=len(L_DECADES_APS), sharey=True, sharex=True)
    for j, (lfm_global, lfm_tc) in enumerate(LFM_COMBS):
        for i, decade in enumerate(L_DECADES_DBLP):
            vals_aps = np.array(results_aps[(lfm_global, lfm_tc)][i])
            vals_dblp = np.array(results_dblp[(lfm_global, lfm_tc)][i])
            if horizontal:
                a_ax[i].barh(
                    [j - .2, j + .2],
                    [vals_dblp[1], vals_aps[1]],
                    height=.4,
                    xerr=[[vals_dblp[1] - vals_dblp[0], vals_dblp[2] - vals_dblp[1]],
                          [vals_aps[1] - vals_aps[0], vals_aps[2] - vals_aps[1]]],
                    color=[MAP_DATA_COLOR[DBLP](.85), MAP_DATA_COLOR[APS](.85)])
            else:
                a_ax[i].bar(
                    [j - .2, j + .2],
                    [vals_dblp[1], vals_aps[1]],
                    width=.4,
                    yerr=[[vals_dblp[1] - vals_dblp[0], vals_dblp[2] - vals_dblp[1]],
                          [vals_aps[1] - vals_aps[0], vals_aps[2] - vals_aps[1]]],
                    color=[MAP_DATA_COLOR[DBLP](.85), MAP_DATA_COLOR[APS](.85)])
            if (lfm_global, lfm_tc) == (CompoundLFM.PAH.value, CompoundLFM.UNIFORM.value):
                # Mark PAH, U with a star
                a_ax[i].text(
                    j + .2,
                    vals_aps[1] + 2,
                    "*",
                    fontweight='bold',
                    color=MAP_DATA_COLOR[APS](.85),
                    verticalalignment='center',
                    horizontalalignment='center',
                )
                a_ax[i].text(
                    j - .2,
                    vals_dblp[1] + 2,
                    "*",
                    fontweight='bold',
                    color=MAP_DATA_COLOR[DBLP](.85),
                    verticalalignment='center',
                    horizontalalignment='center',
                )
        vals_aps = np.array(results_aps[(lfm_global, lfm_tc)][-1])
        if horizontal:
            a_ax[-1].barh(
                j + .2,
                vals_aps[1],
                height=.4,
                xerr=[[vals_aps[1] - vals_aps[0]], [vals_aps[2] - vals_aps[1]]],
                color=MAP_DATA_COLOR[APS](.85))
        else:
            a_ax[-1].bar(
                j + .2,
                vals_aps[1],
                width=.4,
                yerr=[[vals_aps[1] - vals_aps[0]], [vals_aps[2] - vals_aps[1]]],
                color=MAP_DATA_COLOR[APS](.85))
        if (lfm_global, lfm_tc) == (CompoundLFM.PAH.value, CompoundLFM.UNIFORM.value):
            # Mark PAH, U with a star
            a_ax[-1].text(
                j + .2,
                vals_aps[1] + 2,
                "*",
                fontweight='bold',
                color=MAP_DATA_COLOR[APS](.85),
                verticalalignment='center',
                horizontalalignment='center',
            )

    if horizontal:
        a_ax[0].set_yticks(
            range(len(LFM_COMBS)),
            labels=[f"${MAP_LFM_SHORT[lfm_global]},{MAP_LFM_SHORT[lfm_tc]}$"\
                    for lfm_global, lfm_tc in LFM_COMBS])

        for ax in a_ax[1:]:
            ax.spines['left'].set_visible(False)
            ax.tick_params(axis='y', labelleft=False)
        for ax in a_ax:
            ax.set_xlabel('Distance', labelpad=-.5)
    else:
        a_ax[0].set_xticks(
            range(len(LFM_COMBS)),
            labels=[f"{MAP_LFM_SHORT[lfm_global]}\n{MAP_LFM_SHORT[lfm_tc]}"\
                    for lfm_global, lfm_tc in LFM_COMBS])

        for ax in a_ax[1:]:
            ax.spines['left'].set_visible(False)
            ax.tick_params(axis='y', which='both', labelleft=False, left=False)

        a_ax[0].set_ylabel('Distance', labelpad=-.5)

    a_ax[1].legend(
        [plt.Rectangle((0, 0), .4, .4, color=MAP_DATA_COLOR[source](.85)) for source in [DBLP, APS]],
        [str.upper(source) for source in [DBLP, APS]],
        ncols=2,
        loc=(.075, 0.7),
        columnspacing=0.5,
        handlelength=.75,
        labelspacing=0.25,
        handletextpad=0.25)

    # Print decades on top
    for ax, decade in zip(a_ax, L_DECADES_APS):
        ax.set_title(
            f"{decade}" + u"\u2014" + f"{decade + DURATION}", fontsize=10)


            # if i == 0:
            #     # Plot model labels to the right of last markers
            #     y_off = 1.25\
            #         if (lfm_global, lfm_tc) == (CompoundLFM.PAH.value, CompoundLFM.PAH.value)\
            #             else -1.25\
            #                 if (lfm_global, lfm_tc) == (CompoundLFM.HOMOPHILY.value, CompoundLFM.HOMOPHILY.value)\
            #                     else 0
            #     a_ax[i].text(
            #         L_DECADES_DBLP[-1] + 1.5,
            #         values[-1, 1] + y_off,
            #         f"{MAP_LFM_SHORT[lfm_global]},{MAP_LFM_SHORT[lfm_tc]}",
            #         color=MAP_MODEL_COLOR[(lfm_global, lfm_tc)],
            #         verticalalignment='center',
            #         transform=a_ax[i].transData,
            #         # Bold if PAH, U
            #         fontweight='bold'\
            #             if ((lfm_global == CompoundLFM.PAH.value) and (lfm_tc == CompoundLFM.UNIFORM.value))\
            #             else 'normal',
            #         # Italic
            #         fontstyle='italic')



    return fig, a_ax, l_artists

In [ ]:
plot_model_selection(is_subplot=False)

In [ ]:
fig = plt.figure(figsize=(SIZE_FIG[0], SIZE_FIG[1]/3), layout='constrained')
gs = fig.add_gridspec(1, 3)

ax_fm = fig.add_subplot(gs[0, 0])
plot_fm(tpl_fig_ax=(fig, ax_fm))
_=ax_fm.legend(
    # Reduce line length
    # Reduce line width
    handlelength=1.25,
    labelspacing=0.25,
)

ax_sel_dblp = fig.add_subplot(gs[0, -1])
ax_sel_aps = fig.add_subplot(gs[0, 1], sharey=ax_sel_dblp, sharex=ax_sel_dblp)
_, _, l_artists = plot_model_selection(tpl_fig_ax=(fig, [ax_sel_aps, ax_sel_dblp]))
# _ = fig.legend(
#     l_artists,
#     [f"${MAP_LFM_SHORT[lfm_global]},{MAP_LFM_SHORT[lfm_tc]}$" for lfm_global, lfm_tc in LFM_COMBS],
#     loc='center left',
#     bbox_to_anchor=(0.925, 0.6),
#     # Reduce line length
#     # Reduce line width
#     handlelength=1.25,
#     labelspacing=0.25)

ax_sel_dblp.set_ylabel('Distance', rotation=270, labelpad=10)
ax_sel_dblp.yaxis.set_label_position("right")
ax_sel_dblp.yaxis.tick_right()
ax_sel_dblp.spines['right'].set_visible(True)
ax_sel_aps.yaxis.tick_right()
ax_sel_aps.tick_params(axis='y', labelright=False)
ax_sel_aps.spines['left'].set_visible(False)
ax_sel_aps.spines['right'].set_visible(True)


fig.text(0.01, 0.92, "a", fontweight='bold')
fig.text(0.35, 0.92, "b", fontweight='bold')
# fig.tsight_layout()
fig.savefig(os.path.join(
    folder_plots, "model_selection.pdf"), bbox_inches='tight')

## Inequalities overview

In [ ]:
def collect_observed_statistic(
    statistic: str,
    lfm_global: str = CompoundLFM.HOMOPHILY.value,
    lfm_tc: str = CompoundLFM.HOMOPHILY.value,
) -> Tuple[np.ndarray, np.ndarray]:
    a_stats_aps = np.zeros(len(L_DECADES_APS))
    a_stats_dblp = np.zeros(len(L_DECADES_DBLP))
    for source, l_decades, a_stats in zip(
        [APS, DBLP],
        [L_DECADES_APS, L_DECADES_DBLP],
        [a_stats_aps, a_stats_dblp]):
        for j, decade in enumerate(l_decades):
            file_name = create_posterior_file_name(
                create_folder_name(
                    source=source,
                    # Summary stats identical across LFMs
                    lfm_global=lfm_global,
                    lfm_tc=lfm_tc,
                    decade=decade))
            if os.path.exists(file_name):
                with np.load(file_name) as data:
                    a_stats[j] = data[statistic]
    return a_stats_aps, a_stats_dblp

In [ ]:
for lfm_global, lfm_tc in LFM_COMBS:
    a_stat_aps, a_stat_dblp = collect_observed_statistic("mann_whitney", lfm_global, lfm_tc)
    print(f"{lfm_global}, {lfm_tc}")
    print(f"Shapes: {a_stat_aps.shape}, {a_stat_dblp.shape}")
    print(f"APS: {a_stat_aps}")
    print(f"DBLP: {a_stat_dblp}")

In [ ]:
def plot_summary_stat_over_decades(
        statistic: str,
        vneutral: Optional[float] = None,
        tpl_fig_ax: Optional[Tuple[plt.Figure, plt.Axes]] = None,
        is_subplot: bool = True)\
            -> Tuple[plt.Figure, plt.Axes]:
    fig, ax = tpl_fig_ax if tpl_fig_ax is not None else plt.subplots()

    if statistic == "gini_comp":
        a_gini_m_aps, a_gini_m_dblp = collect_observed_statistic("gini_min")
        a_gini_M_aps, a_gini_M_dblp = collect_observed_statistic("gini_maj")
        a_stats_aps = a_gini_m_aps / a_gini_M_aps
        a_stats_dblp = a_gini_m_dblp / a_gini_M_dblp
    elif statistic == "ei":
        a_stats_aps, a_stats_dblp = collect_observed_statistic(statistic)
        # Rescale from [0, 1] to [-1, +1]
        a_stats_aps = 2 * a_stats_aps - 1
        a_stats_dblp = 2 * a_stats_dblp - 1
    else:
        a_stats_aps, a_stats_dblp = collect_observed_statistic(statistic)

    if vneutral is not None:
        ax.axhline(
            vneutral,
            color='black',
            linestyle='--')

    for source, l_decades, a_stats in zip(
            [DBLP, APS],
            [L_DECADES_DBLP, L_DECADES_APS],
            [a_stats_dblp, a_stats_aps]):
        ax.plot(
            l_decades,
            a_stats,
            marker="o",
            color=MAP_DATA_COLOR[source](.85),
            label=str.upper(source))

    ax.set_xlabel(
        "Decade",
        labelpad=-.5)
    # ax.set_ylabel(
    #     MAP_STAT_LABEL[statistic],
    #     labelpad=-5)
    ax.set_title(
        MAP_STAT_LABEL[statistic],
        fontsize=10,
        pad=-.5)


    if not is_subplot:
        _ = ax.legend()

    return fig, ax

In [ ]:
def plot_gini_comp(
        tpl_fig_ax: Optional[Tuple[plt.Figure, plt.Axes]] = None,
        is_subplot: bool = True)\
            -> Tuple[plt.Figure, plt.Axes]:
    fig, ax = tpl_fig_ax if tpl_fig_ax is not None else plt.subplots()

    a_gini_m_aps, a_gini_m_dblp = collect_observed_statistic("gini_min")
    a_gini_M_aps, a_gini_M_dblp = collect_observed_statistic("gini_maj")
    a_gini_cmp_aps = a_gini_m_aps / a_gini_M_aps
    a_gini_cmp_dblp = a_gini_m_dblp / a_gini_M_dblp

    a_mw_aps, a_mw_dblp = collect_observed_statistic("mann_whitney")

    for i, (source, l_decades, a_gini_comp, a_mw) in enumerate(zip(
            [APS, DBLP],
            [L_DECADES_APS, L_DECADES_DBLP],
            [a_gini_cmp_aps, a_gini_cmp_dblp],
            [a_mw_aps, a_mw_dblp])):
        colors = [MAP_DATA_COLOR[source](f) for f in np.linspace(.15, .85, len(L_DECADES_APS))][:len(l_decades)]
        ax.scatter(
            a_mw,
            a_gini_comp,
            marker='o',
            color=colors)

    ax.set_xlabel(
        "Mann-Whitney",
        labelpad=-.5)

    ax.axhline(1, color="black", linestyle="--")
    ax.set_yscale("symlog", linthresh=.01)
    ax.yaxis.tick_right()
    ax.spines['right'].set_visible(True)
    ax.spines['left'].set_visible(False)

    # ax.set_yticks([0.1, 1, 10])
    # ax.set_yticks(np.logspace(np.log10(.1), np.log10(10), 20), minor=True)
    ax.set_ylabel(
        "$\\mathregular{Gini_{m}} / \\mathregular{Gini_{M}}$", labelpad=12)
    # Move y axis right
    ax.yaxis.set_label_position("right")
    ax.yaxis.label.set_rotation(270)

    return fig, ax

In [ ]:
fig = plt.figure(
    figsize=(SIZE_FIG[0], SIZE_FIG[1]/2.75),
    layout='constrained')
gs = fig.add_gridspec(1, 5)

ax_fm = fig.add_subplot(gs[0, 0])
plot_fm(tpl_fig_ax=(fig, ax_fm))
ax_fm.set_ylabel('')
ax_fm.set_title('$f_w$', pad=-.5, fontsize=10)

a_ax = [fig.add_subplot(gs[0, i], sharex=ax_fm)\
        for i in range(1, 5)]
for i, (statistic, vneutral) in enumerate(zip(
        ["ei", "gini", "gini_comp", "mann_whitney"],
        [0., None, 1., 0.5])):
    plot_summary_stat_over_decades(
        statistic,
        vneutral=vneutral,
        tpl_fig_ax=(fig, a_ax[i]))

a_ax[0].legend(
    # Reduce line length
    # Reduce line width
    handlelength=1.25,
    labelspacing=0.125,
    borderpad=0.25,
    handletextpad=0.25,
)

fig.text(0.01, 0.92, "a", fontweight='bold')
fig.text(0.21, 0.92, "b", fontweight='bold')

fig.savefig(os.path.join(
    folder_plots, "summary_stats.pdf"), bbox_inches='tight')

In [ ]:
MODEL_SELECTED_APS = (CompoundLFM.PAH.value, CompoundLFM.UNIFORM.value)
BINS = np.linspace(0, 1, N_BINS + 1)

In [ ]:
def plot_posterior_samples(
        tpl_fig_ax: Optional[Tuple[plt.Figure, plt.Axes]] = None)\
            -> Tuple[plt.Figure, plt.Axes]:
    fig, a_ax = tpl_fig_ax\
        if tpl_fig_ax is not None else\
            plt.subplots(
                ncols=len(L_DECADES_APS),
                nrows=2,
                sharex=True,
                sharey=True)

    a_hist = np.zeros(
        (2, len(L_DECADES_APS), len(BINS) - 1, len(BINS) - 1))
    for i, (source, l_decades) in enumerate(zip([DBLP, APS], [L_DECADES_DBLP, L_DECADES_APS])):
        for j, decade in enumerate(l_decades):
            file_name = create_posterior_file_name(
                create_folder_name(
                    source=source,
                    lfm_global=lfm_global,
                    lfm_tc=lfm_tc,
                    decade=decade))
            if os.path.exists(file_name):
                with np.load(file_name) as data:
                    h = data['h']
                    tau = data['tau']
                    a_hist[i, j] = np.histogram2d(
                        h, tau, density=True, bins=BINS)[0]

    vmin, vmax = np.min(a_hist), np.max(a_hist)
    l_im = []
    for i, (source, l_decades) in enumerate(zip([DBLP, APS], [L_DECADES_DBLP, L_DECADES_APS])):
        for j, decade in enumerate(l_decades):
            im = a_ax[i, j].imshow(
                a_hist[i, j],
                cmap=MAP_DATA_COLOR[source],
                origin='lower',
                extent=(0, 1, 0, 1),
                aspect='equal',
                vmin=vmin, vmax=vmax)
            a_ax[i, j].set_aspect('equal', adjustable='box')
            a_ax[i, j].axhline(
                .5,
                color='black',
                linestyle='--')
        l_im.append(im)

    # x,y = np.meshgrid(
    #     bins[:-1] + np.diff(bins)[0] / 2,
    #     bins[:-1] + np.diff(bins)[0] / 2)
    # a_ax[j].contour(x, y, a_hist[j],
    #                 levels=2,
    #                 cmap='Blues',
    #                 alpha=.5)

    # for ax, decade in zip(a_ax[0], L_DECADES_APS):
    #     ax.set_title(f"{decade}", fontsize=10)

    return fig, a_ax, l_im

In [ ]:
def plot_posterior_gauss(
        tpl_fig_ax: Optional[Tuple[plt.Figure, plt.Axes]] = None)\
            -> Tuple[plt.Figure, plt.Axes]:
    fig, a_ax = tpl_fig_ax\
        if tpl_fig_ax is not None else\
            plt.subplots(
                ncols=len(L_DECADES_APS),
                nrows=2,
                sharex=True,
                sharey=True)

    a_gauss_h, a_gauss_tau = np.zeros(
        (2, 2, len(L_DECADES_APS), N_SAMPLES))
    for i, (source, l_decades) in enumerate(zip([DBLP, APS], [L_DECADES_DBLP, L_DECADES_APS])):
        for j, decade in enumerate(l_decades):
            file_name = create_posterior_file_name(
                create_folder_name(
                    source=source,
                    lfm_global=lfm_global,
                    lfm_tc=lfm_tc,
                    decade=decade))
            if os.path.exists(file_name):
                with np.load(file_name) as data:
                    a_gauss_h[i, j] = data['h']
                    a_gauss_tau[i, j] = data['tau']

    l_im = []
    for i, (source, l_decades) in enumerate(zip([DBLP, APS], [L_DECADES_DBLP, L_DECADES_APS])):
        for j, decade in enumerate(l_decades):
            gauss_h = sp.stats.gaussian_kde(
                a_gauss_h[i, j])
            gauss_tau = sp.stats.gaussian_kde(
                a_gauss_tau[i, j])
            x = np.linspace(0, 1, 100)
            a_ax[j].plot(
                x, gauss_h(x), label=f"{source}", color=MAP_DATA_COLOR[source](.85))
            a_ax[j].fill_between(
                x, gauss_h(x), alpha=.25, color=MAP_DATA_COLOR[source](.85))
            a_ax[j].plot(
                x, -gauss_tau(x), color=MAP_DATA_COLOR[source](.85))
            a_ax[j].fill_between(
                x, -gauss_tau(x), alpha=.25, color=MAP_DATA_COLOR[source](.85))

    for ax in a_ax:
        # Set x-axis at y=0
        ax.spines['bottom'].set_position('zero')
    for ax in a_ax[1:]:
        ax.spines['left'].set_visible(False)
        ax.tick_params(axis='y', labelleft=False, left=False)
        ax.set_ylabel('')

    a_ax[0].set_ylabel(
        'Density',
        # Locate at y=0,
        y=.65)
    a_ax[0].set_yticks(
        [2, 1, 0, -1, -2, -3],
        labels=[f"{abs(x)}".format(x) for x in [2, 1, 0, -1, -2, -3]])

    # Add tau and h labels
    a_ax[0].text(
        .1, -2, "$\\tau$",
        verticalalignment='center')
    a_ax[0].text(
        .1, 2, "$h$",
        verticalalignment='center')

    return fig, a_ax, l_im

In [ ]:
def plot_posterior_overview(
        tpl_fig_ax: Optional[Tuple[plt.Figure, plt.Axes]] = None)\
            -> Tuple[plt.Figure, plt.Axes]:
    fig, ax = tpl_fig_ax\
        if tpl_fig_ax is not None else\
            plt.subplots()

    a_post_h_aps, a_post_tau_aps = np.zeros((2, len(L_DECADES_APS), 3))
    a_post_h_dblp, a_post_tau_dblp = np.zeros((2, len(L_DECADES_DBLP), 3))
    for i, (source, l_decades, a_post_h, a_post_tau) in enumerate(zip(
            [DBLP, APS],
            [L_DECADES_DBLP, L_DECADES_APS],
            [a_post_h_dblp, a_post_h_aps],
            [a_post_tau_dblp, a_post_tau_aps])):
        for j, decade in enumerate(l_decades):
            file_name = create_posterior_file_name(
                create_folder_name(
                    source=source,
                    lfm_global=lfm_global,
                    lfm_tc=lfm_tc,
                    decade=decade))
            if os.path.exists(file_name):
                with np.load(file_name) as data:
                    h = data['h']
                    tau = data['tau']
                    a_post_h[j] = np.quantile(h, q=(0.05, 0.5, 0.95))
                    a_post_tau[j] = np.quantile(tau, q=(0.05, 0.5, 0.95))

    for i, (source, l_decades, a_post_h, a_post_tau) in enumerate(zip(
            [DBLP, APS],
            [L_DECADES_DBLP, L_DECADES_APS],
            [a_post_h_dblp, a_post_h_aps],
            [a_post_tau_dblp, a_post_tau_aps])):
        for j, decade in enumerate(l_decades):
            ax.errorbar(
                l_decades,
                a_post_h[:, 1],
                xerr=a_post_h[:, 1] - a_post_h[:, 0],
                yerr=a_post_h[:, 2] - a_post_h[:, 1],
                fmt='s-',
                color=MAP_DATA_COLOR[source](.85),
                alpha=.5)
            ax.errorbar(
                l_decades,
                a_post_tau[:, 1],
                xerr=a_post_tau[:, 1] - a_post_tau[:, 0],
                yerr=a_post_tau[:, 2] - a_post_tau[:, 1],
                fmt='^-',
                color=MAP_DATA_COLOR[source](.85))

    return fig, ax

In [ ]:
fig = plt.figure(
    figsize=(SIZE_FIG[0], SIZE_FIG[1]*(.9)),
    constrained_layout=True
    )
gs = fig.add_gridspec(
    nrows=2,
    height_ratios=[1, 2]
)

gs_model_select = gs[0].subgridspec(
    nrows=1,
    ncols=len(L_DECADES_APS))

# Plot model selection
a_ax_model_select = [
    fig.add_subplot(
        gs_model_select[0])]
for i, decade in enumerate(L_DECADES_APS[1:], start=1):
    a_ax_model_select.append(fig.add_subplot(
        gs_model_select[i],
        sharex=a_ax_model_select[0],
        sharey=a_ax_model_select[0]))
_ = plot_model_selection_bar(
    tpl_fig_ax=(fig, a_ax_model_select),
    horizontal=False)

# DBLP
gs_posteriors = gs[1].subgridspec(
    nrows=1,
    ncols=1+len(L_DECADES_APS),
    width_ratios=[0.05, 1, 1, 1, 1, 1])
# Plot posterior samples
a_ax_posteriors = [
    fig.add_subplot(gs_posteriors[1])]
for i, decade in enumerate(L_DECADES_APS[1:], start=2):
    a_ax_posteriors.append(fig.add_subplot(
        gs_posteriors[i],
        sharey=a_ax_posteriors[0],
        sharex=a_ax_posteriors[0]))
a_ax_posteriors = np.asarray(a_ax_posteriors)
_, _, l_im = plot_posterior_gauss(
    tpl_fig_ax=(
        fig, a_ax_posteriors))

fig.text(0.02, 0.98, "a", fontweight='bold')
fig.text(0.02, 0.65, "b", fontweight='bold')

fig.text(0.045, .6825, "$\\mathcal{{L}}_G\colon$\n$\\mathcal{{L}}_T\colon$")

fig.savefig(os.path.join(
    folder_plots, "inference_gauss.pdf"), bbox_inches='tight')


In [ ]:
fig = plt.figure(
    figsize=(SIZE_FIG[0], SIZE_FIG[1]*(.9)),
    constrained_layout=True
    )
gs = fig.add_gridspec(
    nrows=3,
    height_ratios=[1, 1, 1]
)

gs_model_select = gs[0].subgridspec(
    nrows=1,
    ncols=4,
    width_ratios=[1/6, 1/3, 1/3, 1/6],)

# Plot model selection
ax_model_selection_dblp = fig.add_subplot(
    gs_model_select[1])

ax_model_selection_aps = fig.add_subplot(
    gs_model_select[2],
    sharex=ax_model_selection_dblp,
    sharey=ax_model_selection_dblp)
_ = plot_model_selection(
    tpl_fig_ax=(fig, [ax_model_selection_dblp, ax_model_selection_aps]))

ax_model_selection_dblp.set_ylabel(
    'Distance')
ax_model_selection_aps.tick_params(axis='y', labelleft=False)

# Plot posterior samples
a_ax_posterior = [[], []]

# DBLP
gs_posterior_dblp = gs[1].subgridspec(
    nrows=1,
    ncols=6,
    width_ratios=[1, 1, 1, 1, .05, 1])
a_ax_posterior[0].append(fig.add_subplot(
    gs_posterior_dblp[0]))
for i, decade in enumerate(L_DECADES_DBLP[1:], start=1):
    a_ax_posterior[0].append(fig.add_subplot(
        gs_posterior_dblp[i],
        sharey=a_ax_posterior[0][0],
        sharex=a_ax_posterior[0][0]))
# Add empty plot to DBLP
a_ax_posterior[0].append(None)

# APS
gs_posterior_aps = gs[2].subgridspec(
    nrows=1,
    ncols=6,
    width_ratios=[1, 1, 1, 1, 1, .05])
a_ax_posterior[1].append(fig.add_subplot(
    gs_posterior_aps[0],
    sharex=a_ax_posterior[0][0],
    sharey=a_ax_posterior[0][0]))
for i, decade in enumerate(L_DECADES_APS[1:], start=1):
    a_ax_posterior[1].append(fig.add_subplot(
        gs_posterior_aps[i],
        sharey=a_ax_posterior[0][0],
        sharex=a_ax_posterior[0][0]))

a_ax_posterior = np.asarray(a_ax_posterior)
_, _, l_im = plot_posterior_samples(
    tpl_fig_ax=(
        fig, a_ax_posterior))

for ax in a_ax_posterior[:, 0]:
    ax.set_ylabel(
        r"$h$",
        labelpad=-5)
    ax.set_yticks(
        [0, 1])
for ax in a_ax_posterior[-1].flatten():
    if ax is None:
        continue
    ax.set_xlabel(r"$\tau$",
        # Reduce distance to axis
        labelpad=-5)
    ax.set_xticks(
        [0, 1])

for i, decade in enumerate(L_DECADES_DBLP):
    _t = a_ax_posterior[0, i].set_title(
        f"{decade}",
        fontsize=10,
        pad=-.5)
# Add another year to the right of the last title at same height
fig.text(
    _t.get_position()[0] + 1.3,
    _t.get_position()[1] + .027,
    L_DECADES_APS[-1],
    transform=a_ax_posterior[0, -2].transAxes,
)

# Hide y-axis labels for all but the first column
for ax in a_ax_posterior[:, 1:].flatten():
    if ax is None:
        continue
    ax.tick_params(axis='y', labelleft=False)
    ax.set_ylabel('')

# Add colorbars
ax_cb_dblp = fig.add_subplot(
    gs_posterior_dblp[-2])
cbar_dblp = fig.colorbar(
    l_im[0],
    cax=ax_cb_dblp,
    orientation='vertical',)
cbar_dblp.set_label('PDE', loc='top', rotation=0, labelpad=8.5)

ax_cb_aps = fig.add_subplot(
    gs_posterior_aps[-1])
cbar_aps = fig.colorbar(
    l_im[1],
    cax=ax_cb_aps,
    orientation='vertical')
cbar_aps.set_label('PDE', loc='top', rotation=0, labelpad=8.5)

fig.text(0.02, 0.98, "a", fontweight='bold')
fig.text(0.02, 0.67, "b", fontweight='bold')

fig.savefig(os.path.join(
    folder_plots, "inference.pdf"), bbox_inches='tight')


In [ ]:
fig = plt.figure(
    figsize=(SIZE_FIG[0], SIZE_FIG[1]*(0.9)),
    constrained_layout=True
    )
gs = fig.add_gridspec(
    nrows=3,
    height_ratios=[1, 1, 1]
)

gs_model_select = gs[0].subgridspec(
    nrows=1,
    ncols=2)

# Plot model selection
ax_model_selection_dblp = fig.add_subplot(
    gs_model_select[0])

ax_model_selection_aps = fig.add_subplot(
    gs_model_select[1],
    sharex=ax_model_selection_dblp,
    sharey=ax_model_selection_dblp)
_ = plot_model_selection(
    tpl_fig_ax=(fig, [ax_model_selection_dblp, ax_model_selection_aps]))

ax_model_selection_dblp.set_ylabel(
    'Distance')
ax_model_selection_aps.tick_params(axis='y', labelleft=False)

# Set axis and label to the right side
# ax_model_selection_dblp.yaxis.tick_right()
# ax_model_selection_dblp.yaxis.set_label_position("right")
# ax_model_selection_dblp.spines['right'].set_visible(True)
# ax_model_selection_dblp.spines['left'].set_visible(False)

# Hide y-axis
# ax_model_selection_aps.spines['left'].set_visible(False)
# ax_model_selection_aps.spines['right'].set_visible(True)
# ax_model_selection_aps.yaxis.tick_right()
# # Remove tick labels on y-axis for APS
# # ax_model_selection_aps.set_yticklabels([])
# # Remove y-axis label
# ax_model_selection_aps.set_ylabel('')

# Plot overview
# ax_overview = fig.add_subplot(
#     gs_model_select[-1],
#     sharex=ax_model_selection_dblp)
# ax_overview.set_xlabel(
#     "Decade",
#     labelpad=-.5)
# plot_posterior_overview(
#     tpl_fig_ax=(fig, ax_overview))


# ax_model_selection_legend = fig.add_subplot(
#     gs[0,
#        _x[0][4]:_x[0][5]])

# legend = ax_model_selection_legend.legend(
#     l_artists,
#     [f"${MAP_LFM_SHORT[lfm_global]},{MAP_LFM_SHORT[lfm_tc]}$" for lfm_global, lfm_tc in LFM_COMBS],
#     loc='center',
#     # bbox_to_anchor=(0.975, 0.5),
#     # Reduce line length
#     # Reduce line width
#     handlelength=1.125,
#     labelspacing=0.125)
# legend.set_title("Model")
# ax_model_selection_legend.axis('off')

# Plot posterior samples
a_ax_posterior = [[], []]

# DBLP
gs_posterior_dblp = gs[1].subgridspec(
    nrows=1,
    ncols=6,
    width_ratios=[1, 1, 1, 1, .05, 1])
a_ax_posterior[0].append(fig.add_subplot(
    gs_posterior_dblp[0]))
for i, decade in enumerate(L_DECADES_DBLP[1:], start=1):
    a_ax_posterior[0].append(fig.add_subplot(
        gs_posterior_dblp[i],
        sharey=a_ax_posterior[0][0],
        sharex=a_ax_posterior[0][0]))
# Add empty plot to DBLP
a_ax_posterior[0].append(None)

# APS
gs_posterior_aps = gs[2].subgridspec(
    nrows=1,
    ncols=6,
    width_ratios=[1, 1, 1, 1, 1, .05])
a_ax_posterior[1].append(fig.add_subplot(
    gs_posterior_aps[0],
    sharex=a_ax_posterior[0][0],
    sharey=a_ax_posterior[0][0]))
for i, decade in enumerate(L_DECADES_APS[1:], start=1):
    a_ax_posterior[1].append(fig.add_subplot(
        gs_posterior_aps[i],
        sharey=a_ax_posterior[0][0],
        sharex=a_ax_posterior[0][0]))

a_ax_posterior = np.asarray(a_ax_posterior)
_, _, l_im = plot_posterior_samples(
    tpl_fig_ax=(
        fig, a_ax_posterior))

# a_ax_posterior[1][-1].plot(
#     L_DECADES_APS,
#     np.zeros(len(L_DECADES_APS)),
# )

# Hide x-axis labels for all but the last row
# for ax in a_ax_posterior.flatten():
#     if ax is None: continue
#     ax.tick_params(axis='x', labelbottom=False)
#     ax.set_xticks(
#         [0, 1]
#     )
#     ax.set_xlabel('')
# a_ax_posterior[0, -1].set_xlabel(
#     r"$\tau$",
#     labelpad=-5)

for ax in a_ax_posterior[:, 0]:
    ax.set_ylabel(
        r"$h$",
        labelpad=-5)
    ax.set_yticks(
        [0, 1])
for ax in a_ax_posterior[-1].flatten():
    if ax is None:
        continue
    ax.set_xlabel(r"$\tau$",
        # Reduce distance to axis
        labelpad=-5)
    ax.set_xticks(
        [0, 1])

for i, decade in enumerate(L_DECADES_DBLP):
    _t = a_ax_posterior[0, i].set_title(
        f"{decade}",
        fontsize=10,
        pad=-.5)
# Add another year to the right of the last title at same height
fig.text(
    _t.get_position()[0] + 1.3,
    _t.get_position()[1] + .027,
    L_DECADES_APS[-1],
    transform=a_ax_posterior[0, -2].transAxes,
)

# Hide y-axis labels for all but the first column
for ax in a_ax_posterior[:, 1:].flatten():
    if ax is None:
        continue
    ax.tick_params(axis='y', labelleft=False)
    ax.set_ylabel('')

# Remove axis for last plot
# a_ax_posterior[1, -1].axis('off')

# Add colorbars
ax_cb_dblp = fig.add_subplot(
    gs_posterior_dblp[-2])
cbar_dblp = fig.colorbar(
    l_im[0],
    cax=ax_cb_dblp,
    orientation='vertical',)

ax_cb_aps = fig.add_subplot(
    gs_posterior_aps[-1])
cbar_aps = fig.colorbar(
    l_im[1],
    cax=ax_cb_aps,
    orientation='vertical')
cbar_aps.set_label('PDE', loc='top', rotation=0, labelpad=7.5)


# Adjust layout manually
# fig.subplots_adjust(
#     hspace=.35,
#     wspace=.25)

fig.text(0.02, 0.98, "a", fontweight='bold')
fig.text(0.02, 0.67, "b", fontweight='bold')
# fig.text(0.666, 0.98, "c", fontweight='bold')

# fig.tight_layout()
fig.savefig(os.path.join(
    folder_plots, "inference.pdf"), bbox_inches='tight')


In [ ]:
SIZE_FIG[0] /SIZE_FIG[1]

In [ ]:
SIZE_FIG[1]

## Summary statistics

In [ ]:
FOLDER_ARRAY_POOL = "summary_stats_sim"
# L_METRICS = ["ccf_0", "ccf_1", "ccf_2", "ccf_3", "ccf_4", "ccf_5", "ccf_6", "ccf_7", "ei", "gini", "gini_maj", "gini_min", "mann_whitney"]
L_METRICS = ["mean_ccf", "ei", "gini", "gini_maj", "gini_min", "mann_whitney"]
# L_METRICS = ["ccf_0", "ccf_3", "ccf_6", "ccf_7", "ei", "gini", "gini_maj", "gini_min", "mann_whitney"]

In [ ]:
d_metrics = {
    metric: np.load(
        os.path.join(create_folder_name(
            lfm_global=CompoundLFM.HOMOPHILY.value,
            lfm_tc=CompoundLFM.UNIFORM.value),
            FOLDER_ARRAY_POOL, f"{metric}.npy"))\
    for metric in L_METRICS}

In [ ]:
a_posteriors = np.load(
    os.path.join(create_folder_name(
        lfm_global=CompoundLFM.PAH.value,
        lfm_tc=CompoundLFM.PAH.value), "posteriors.npz")
)
print("Observed metrics:")
for metric in L_METRICS:
    print(f"{metric}_obs: {a_posteriors[metric]}")

In [ ]:
print("Mean and std of simulations:")
for metric, data in d_metrics.items():
    print(f"{metric}: {np.mean(data)}, {np.std(data)}")

In [ ]:
# Plot metric distributions in a grid
fig, axes = plt.subplots(
    3, 5, figsize=(20, 12))
for i, metric in enumerate(L_METRICS):
    ax = axes.flatten()[i]
    data = d_metrics[metric]
    ax.hist(data, bins=25, density=True)
    ax.axvline(a_posteriors[metric], color='r', linestyle='--')
    ax.set_title(metric)
fig.tight_layout()